# Modelling - SEMMA

In [21]:
import sqlite3
import pandas as pd
from pathlib import Path
import numpy as np

## SAMPLE (Amostragem)

### Extraindo  os Dados do Banco

In [ ]:
DB_PRINCIPAL = "../../data/processed/feature_store.db"   
DB_DIVIDAS   = "../../data/processed/database.db"         

query = Path("target.sql").read_text(encoding="utf-8")

con = sqlite3.connect(DB_PRINCIPAL)

try:
    con.execute(f"ATTACH DATABASE '{DB_DIVIDAS}' AS dividas")
    df = pd.read_sql_query(query, con)
finally:
    con.close()

print(df.shape)
print(df["target_ever90_mob12"].mean())

(104623, 299)
0.0347820268965715


In [3]:
df.to_parquet('../../data/processed/abt.parquet')

### Split de Treinamento

In [3]:
df = pd.read_parquet('../../data/processed/abt.parquet')

In [11]:
df['data_ref'] = pd.to_datetime(df['data_ref'])

data_corte = '2023-06-01'

df_train = df[df['data_ref'] < data_corte].copy()
df_oot = df[df['data_ref'] >= data_corte].copy()

print(f"Dimensões Treino: {df_train.shape}")
print(f"Dimensões OOT: {df_oot.shape}")

print(f"Interval de Tempo Treino: {df_train['data_ref'].min()} até {df_train['data_ref'].max()}")
print(f"Interval de Tempo OOT: {df_oot['data_ref'].min()} até {df_oot['data_ref'].max()}")

print(f"Inadimplência Treino: {df_train['target_ever90_mob12'].mean() * 100:.2f}%")
print(f"Inadimplência OOT: {df_oot['target_ever90_mob12'].mean() * 100:.2f}%")


Dimensões Treino: (82836, 299)
Dimensões OOT: (21787, 299)
Interval de Tempo Treino: 2017-02-04 00:00:00 até 2023-05-31 00:00:00
Interval de Tempo OOT: 2023-06-01 00:00:00 até 2024-03-05 00:00:00
Inadimplência Treino: 3.22%
Inadimplência OOT: 4.45%


In [7]:
X_train = df_train.drop(columns=['target_ever90_mob12'])
y_train = df_train['target_ever90_mob12']

X_oot = df_oot.drop(columns=['target_ever90_mob12'])
y_oot = df_oot['target_ever90_mob12']

## Explore (Exploração)

### Variáveis Númericas

In [15]:
correlations = X_train.select_dtypes(include=['number']).corrwith(y_train)
correlations.sort_values(ascending=False).head(20)


c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


flag_cliente_sem_historico          0.049172
flag_credito_acima_max_historico    0.029530
pct_recusados_30d                   0.027646
flag_primeira_vez_tipo_produto      0.026071
pct_pagamentos_parciais             0.021993
pct_atraso_5d_3m                    0.020226
pct_recusados_120d                  0.018311
dias_desde_ultimo_pagamento         0.017548
pct_recusados_90d                   0.017458
pct_recusa_desistencia_cliente      0.016628
flag_novo_tipo_contrato             0.016611
flag_primeira_vez_tipo_contrato     0.016611
variacao_parcela_vs_media_6m        0.016146
pct_atraso_5d_6m                    0.016030
dias_desde_ultimo_contrato          0.015325
pct_parcelas_atrasadas_6m           0.015290
dias_desde_ultimo_aprovado          0.014991
pct_recusados_6m                    0.014703
desvio_credito_vs_ultimo            0.013841
pct_recusados_12m                   0.013547
dtype: float64

In [20]:
correlations = X_train.select_dtypes(include=['number']).corrwith(y_train)
correlations.sort_values(ascending=True).head(20)

c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\patry\AppData\Local\miniconda3\envs\case_datarisk\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


pct_aprovados_30d                -0.052660
qtd_contratos_anteriores         -0.047438
qtd_aprovados                    -0.047438
qtd_parcelas_vencidas_vida       -0.044894
qtd_total                        -0.044514
qtd_areas_distintas              -0.043500
qtd_parcelas_vencidas_12m        -0.043426
qtd_parcelas_vencidas_6m         -0.041278
razao_pago_previsto              -0.039679
qtd_parcelas_vencidas_3m         -0.039532
prazo_restante_medio_aberto      -0.038437
qtd_contratos_ativos_estimado    -0.038151
qtd_contratos_abertos_parc       -0.037886
qtd_produto_nao_informado        -0.037798
qtd_tipos_produto_distintos      -0.036613
qtd_tipo_emprestimo_dinheiro     -0.035482
qtd_tipo_financiamento_consumo   -0.035033
qtd_aprovados_12m                -0.033513
valor_credito_soma               -0.033019
qtd_tipo_credito_rotativo        -0.032028
dtype: float64

In [ ]:

df_train_explore = X_train.copy()
df_train_explore["target"] = y_train

describe = (
    df_train_explore
    .groupby("target")
    .mean(numeric_only=True)
    .T
)

describe["variable"] = describe.index
describe["ratio"] = describe[1] / describe[0].replace(0, np.nan)

cols = ["variable"] + [col for col in describe.columns if col != "variable"]
describe = describe[cols]


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', None)
display(describe)
     

### Variáveis Categóricas

In [28]:
df_train_explore.select_dtypes(exclude=['number']).columns

Index(['data_ref', 'tipo_contrato', 'dia_semana', 'faixa_horaria', 'status_contrato', 'estado_civil', 'tipo_renda', 'nivel_educacao', 'ocupacao', 'tipo_organizacao', 'tipo_moradia'], dtype='object')

In [31]:
print("Frequência relativa de tipo_contrato para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['tipo_contrato'].value_counts(normalize=True))

print("\nFrequência relativa de tipo_contrato para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['tipo_contrato'].value_counts(normalize=True))

Frequência relativa de tipo_contrato para target = 0:


tipo_contrato
Empréstimo em Dinheiro      0.377799
Financiamento de Consumo    0.368855
Crédito Rotativo            0.253346
Name: proportion, dtype: float64


Frequência relativa de tipo_contrato para target = 1:


tipo_contrato
Empréstimo em Dinheiro      0.389284
Financiamento de Consumo    0.377670
Crédito Rotativo            0.233046
Name: proportion, dtype: float64

In [33]:
print("Frequência relativa de faixa_horaria para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['faixa_horaria'].value_counts(normalize=True))

print("\nFrequência relativa de faixa_horaria para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['faixa_horaria'].value_counts(normalize=True))

Frequência relativa de faixa_horaria para target = 0:


faixa_horaria
tarde        0.532301
manha        0.378909
noite        0.074470
madrugada    0.014320
Name: proportion, dtype: float64


Frequência relativa de faixa_horaria para target = 1:


faixa_horaria
tarde        0.544773
manha        0.364181
noite        0.074934
madrugada    0.016111
Name: proportion, dtype: float64

In [ ]:
print("Frequência relativa de estado_civil para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['estado_civil'].value_counts(normalize=True))

print("\nFrequência relativa de estado_civil para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['estado_civil'].value_counts(normalize=True))

Frequência relativa de estado_civil para target = 0:


estado_civil
Casado           0.652189
Solteiro         0.129442
União estável    0.095750
Separado         0.066411
Viúvo            0.056208
Name: proportion, dtype: float64


Frequência relativa de estado_civil para target = 1:


estado_civil
Casado           0.636943
Solteiro         0.141251
União estável    0.097415
Separado         0.067066
Viúvo            0.057325
Name: proportion, dtype: float64

In [36]:
print("Frequência relativa de tipo_renda para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['tipo_renda'].value_counts(normalize=True))

print("\nFrequência relativa de tipo_renda para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['tipo_renda'].value_counts(normalize=True))

Frequência relativa de tipo_renda para target = 0:


tipo_renda
Assalariado            0.496988
Associado comercial    0.227438
Aposentado             0.195554
Servidor público       0.072025
Autônomo               0.006212
Não informado          0.001647
Desempregado           0.000075
Licença maternidade    0.000050
Estudante              0.000012
Name: proportion, dtype: float64


Frequência relativa de tipo_renda para target = 1:


tipo_renda
Assalariado            0.531285
Associado comercial    0.206819
Aposentado             0.180217
Servidor público       0.075309
Autônomo               0.005620
Não informado          0.000749
Name: proportion, dtype: float64

In [37]:
print("Frequência relativa de nivel_educacao para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['nivel_educacao'].value_counts(normalize=True))

print("\nFrequência relativa de nivel_educacao para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['nivel_educacao'].value_counts(normalize=True))

Frequência relativa de nivel_educacao para target = 0:


nivel_educacao
Ensino médio                  0.736537
Ensino superior               0.216486
Ensino superior incompleto    0.032719
Ensino fundamental            0.013060
Pós-graduação                 0.001198
Name: proportion, dtype: float64


Frequência relativa de nivel_educacao para target = 1:


nivel_educacao
Ensino médio                  0.724991
Ensino superior               0.230049
Ensino superior incompleto    0.032222
Ensino fundamental            0.010865
Pós-graduação                 0.001873
Name: proportion, dtype: float64

In [38]:
print("Frequência relativa de ocupacao para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['ocupacao'].value_counts(normalize=True))

print("\nFrequência relativa de ocupacao para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['ocupacao'].value_counts(normalize=True))

Frequência relativa de ocupacao para target = 0:


ocupacao
Operários                          0.180087
Aposentado                         0.177517
Não informado                      0.146157
Equipe de vendas                   0.105729
Equipe administrativa              0.085559
Gerentes                           0.068407
Motoristas                         0.060000
Técnicos especializados            0.038532
Contadores                         0.032557
Profissionais de saúde             0.030299
Equipe de segurança                0.019883
Cozinheiros                        0.017414
Equipe de limpeza                  0.013659
Serviços particulares              0.007285
Operários de baixa qualificação    0.004865
Secretárias                        0.003468
Garçons e atendentes de bar        0.003318
Corretores imobiliários            0.002208
Equipe de tecnologia               0.001509
Equipe de recursos humanos         0.001472
Sem Ocupação                       0.000075
Name: proportion, dtype: float64


Frequência relativa de ocupacao para target = 1:


ocupacao
Operários                          0.189959
Aposentado                         0.161109
Não informado                      0.154365
Equipe de vendas                   0.107156
Equipe administrativa              0.092919
Gerentes                           0.064444
Motoristas                         0.057700
Técnicos especializados            0.032596
Profissionais de saúde             0.031472
Contadores                         0.026227
Equipe de segurança                0.022855
Cozinheiros                        0.016111
Equipe de limpeza                  0.013488
Serviços particulares              0.009741
Garçons e atendentes de bar        0.005245
Operários de baixa qualificação    0.005245
Secretárias                        0.003747
Corretores imobiliários            0.002623
Equipe de recursos humanos         0.001873
Equipe de tecnologia               0.001124
Name: proportion, dtype: float64

In [39]:
print("Frequência relativa de tipo_organizacao para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['tipo_organizacao'].value_counts(normalize=True))

print("\nFrequência relativa de tipo_organizacao para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['tipo_organizacao'].value_counts(normalize=True))

Frequência relativa de tipo_organizacao para target = 0:


tipo_organizacao
Entidade empresarial tipo 3    0.219317
Não informado                  0.207654
Autônomo                       0.124378
Outro                          0.057630
Governo                        0.041750
Saúde                          0.041488
Escola                         0.034890
Entidade empresarial tipo 2    0.029987
Comércio tipo 7                0.029226
Educação infantil              0.019422
Construção                     0.018187
Entidade empresarial tipo 1    0.016553
Transporte tipo 4              0.014782
Segurança                      0.009568
Indústria tipo 3               0.008557
Indústria tipo 9               0.008358
Habitação                      0.008071
Comércio tipo 3                0.007933
Indústria tipo 11              0.007884
Correios                       0.007023
Polícia                        0.006773
Transporte tipo 2              0.006699
Forças armadas                 0.006636
Agricultura                    0.005364
Banco                  


Frequência relativa de tipo_organizacao para target = 1:


tipo_organizacao
Entidade empresarial tipo 3    0.204196
Não informado                  0.191083
Autônomo                       0.134882
Outro                          0.059198
Governo                        0.044961
Saúde                          0.042713
Escola                         0.040090
Comércio tipo 7                0.028475
Entidade empresarial tipo 2    0.024354
Educação infantil              0.019858
Construção                     0.019108
Entidade empresarial tipo 1    0.015362
Transporte tipo 4              0.014987
Segurança                      0.010865
Forças armadas                 0.010491
Indústria tipo 11              0.010491
Habitação                      0.008617
Indústria tipo 9               0.008243
Transporte tipo 2              0.007868
Restaurante                    0.007493
Correios                       0.007119
Banco                          0.006744
Comércio tipo 3                0.006744
Polícia                        0.006744
Indústria tipo 3       

In [40]:
print("Frequência relativa de tipo_moradia para target = 0:")
display(df_train_explore[df_train_explore['target'] == 0]['tipo_moradia'].value_counts(normalize=True))

print("\nFrequência relativa de tipo_moradia para target = 1:")
display(df_train_explore[df_train_explore['target'] == 1]['tipo_moradia'].value_counts(normalize=True))

Frequência relativa de tipo_moradia para target = 0:


tipo_moradia
Casa ou apartamento        0.894171
Mora com os pais           0.042424
Apartamento municipal      0.036037
Imóvel alugado             0.015917
Apartamento funcional      0.008557
Apartamento cooperativo    0.002894
Name: proportion, dtype: float64


Frequência relativa de tipo_moradia para target = 1:


tipo_moradia
Casa ou apartamento        0.896590
Mora com os pais           0.044961
Apartamento municipal      0.031847
Imóvel alugado             0.015362
Apartamento funcional      0.009367
Apartamento cooperativo    0.001873
Name: proportion, dtype: float64